In [2]:
from transformers import AutoTokenizer, AutoModel

slug = 'distilbert/distilbert-base-uncased'

In [ ]:
import huggingface_hub as hfh
from datasets import load_dataset
hfh.login()

dsname = 'cornell-movie-review-data/rotten_tomatoes'

full_dataset = load_dataset(path=dsname, cache_dir='./mydatasets')
trainset = full_dataset['train']
validset = full_dataset['validation']


BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer()

def data_process(sample):
    return tokenizer(sample['text'], max_length=512, padding='max_length')

trainset = trainset.map(data_process)
trainset.set_format('pt', columns=['input_ids'], output_all_columns=True)
print(trainset[0])

Map: 100%|██████████| 8530/8530 [00:01<00:00, 4781.39 examples/s]

{'input_ids': tensor([2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 

In [12]:
from transformers import DistilBertForSequenceClassification
model = DistilBertForSequenceClassification.from_pretrained(slug, num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11992.29it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
model(trainset[0]['input_ids'])

SequenceClassifierOutput(loss=None, logits=tensor([[0.0587, 0.0558]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [15]:
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer

dc = DataCollatorWithPadding(tokenizer)

train_args = TrainingArguments(
    output_dir='./output',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    logging_strategy='epoch',
    logging_steps=1,
)

trainer = Trainer(model=model, args=train_args, train_dataset=trainset, data_collator=dc)
trainer.train()

Step,Training Loss
1067,0.697008
2134,0.696454
3201,0.695223
4268,0.695149
5335,0.693713


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.37it/s]


TrainOutput(global_step=5335, training_loss=0.6955093455292292, metrics={'train_runtime': 588.2428, 'train_samples_per_second': 72.504, 'train_steps_per_second': 9.069, 'total_flos': 5649734552678400.0, 'train_loss': 0.6955093455292292, 'epoch': 5.0})